# 02 Â· ASVspoof 5 â€” XLSR-MamBo

Primo esperimento allo stato dell'arte sui subset preparati nel notebook 01 (forma d'onda grezza, non le Mel).

**Modello**: XLSR-MamBo, configurazione MamBo-3-Hydra-N3 (Ng et al., *XLSR-MamBo: Scaling the Hybrid
Mamba-Attention Backbone for Audio Deepfake Detection*, arXiv 2601.02944, ACL 2026 Findings;
codice originale https://github.com/saki-ciallo/MamBo-for-ADD, MIT). XLS-R 300M fine-tunato â†’
5 unitÃ  [3Ã—Hydra, MLP, MHA, MLP] â†’ attention pooling â†’ 2 classi. Port in PyTorch puro in `src/thesis/models/`.

**Protocollo**: train sul subset train (5000+5000), scelta dell'epoca **solo** sull'EER del dev
(2500+2500, attacchi A09â€“A16), test una sola volta sull'eval (5000+5000, attacchi A17â€“A32 e codec mai visti).
3 seed; metriche EER, minDCF, actDCF, Cllr, EER per attacco e per codec, IC 95% bootstrap dell'EER.

**Differenze dal paper** (dovute alla 3060 da 8 GB / Windows): XLS-R da Hugging Face invece che da torchaudio
(stessi pesi); CNN iniziale di XLS-R congelata e gradient checkpointing; batch 8 Ã— 4 di accumulo (32 effettivo);
focal loss con Î± = 0.5/0.5 perchÃ© i subset sono bilanciati; nessuna RawBoost (esperimento separato);
training sui subset di ASVspoof 5, non su ASVspoof 2019 LA.

Il training gira **fuori da Jupyter** (`python -m thesis.train`), cosÃ¬ non si ferma se VS Code si scollega.
Questo notebook lo lancia e poi analizza i risultati salvati su file.

## 0 Â· Setup

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

REPO = Path.cwd().resolve().parent  # il notebook gira da notebooks/
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from thesis import config
from thesis.train import TrainConfig

print(config.describe())

## 1 Â· Configurazione

In [ ]:
CONFIG = REPO / "configs" / "mambo_asvspoof5.json"
SEEDS = [1, 2, 3]

cfg = TrainConfig.from_json(CONFIG)
RUN_DIR = config.ARTIFACTS_ROOT / "runs" / cfg.name
print(json.dumps(json.loads(CONFIG.read_text()), indent=2))
print("Risultati in:", RUN_DIR)

## 2 Â· Lancio del training in background

La cella avvia un processo separato e torna subito: il training continua anche chiudendo VS Code.
I seed giÃ  completati (con `metrics.json`) vengono saltati, quindi si puÃ² rilanciare senza perdere nulla.
Il log testuale Ã¨ in `RUN_DIR/train.log`. **Lanciare una sola volta**: due training insieme non stanno in 8 GB.

In [ ]:
LAUNCH = False  # metti True per avviare

if LAUNCH:
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    log = open(RUN_DIR / "train.log", "a", encoding="utf-8")
    flags = 0
    if sys.platform == "win32":  # processo staccato dal kernel e da VS Code
        flags = subprocess.DETACHED_PROCESS | subprocess.CREATE_NEW_PROCESS_GROUP | subprocess.CREATE_BREAKAWAY_FROM_JOB
    proc = subprocess.Popen(
        [sys.executable, "-m", "thesis.train", "--config", str(CONFIG), "--seed", *map(str, SEEDS)],
        cwd=REPO, stdout=log, stderr=subprocess.STDOUT, creationflags=flags,
        env={**os.environ, "PYTHONPATH": str(REPO / "src")},
    )
    print("Avviato, PID", proc.pid)

## 3 Â· Avanzamento

In [ ]:
for seed in SEEDS:
    log_path = RUN_DIR / f"seed{seed}" / "log.csv"
    done = (RUN_DIR / f"seed{seed}" / "metrics.json").exists()
    if log_path.exists():
        log = pd.read_csv(log_path)
        last = log.iloc[-1]
        print(f"seed {seed}: {'completato' if done else 'in corso'} â€” epoca {int(last.epoca)}, "
              f"dev EER {last.dev_eer:.4f}, {last.sec_totale / 60:.0f} min/epoca, GPU max {last.gpu_max_gb} GB")
    else:
        print(f"seed {seed}: non ancora iniziato")

## 4 Â· Curve di training

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for seed in SEEDS:
    log_path = RUN_DIR / f"seed{seed}" / "log.csv"
    if not log_path.exists():
        continue
    log = pd.read_csv(log_path)
    axes[0].plot(log.epoca, log.train_loss, label=f"train s{seed}")
    axes[0].plot(log.epoca, log.dev_loss, "--", label=f"dev s{seed}")
    axes[1].plot(log.epoca, 100 * log.dev_eer, marker="o", label=f"seed {seed}")
axes[0].set(title="Loss (train: focal, dev: cross-entropy)", xlabel="epoca"); axes[0].legend(fontsize=7)
axes[1].set(title="EER dev (%)", xlabel="epoca"); axes[1].legend()
plt.tight_layout()

## 5 Â· Risultati

Tabella generata dai `metrics.json` (media Â± deviazione standard sui seed).
actDCF e Cllr misurano la calibrazione dei punteggi (logit bonafide âˆ’ logit spoof, senza calibrazione a posteriori).

In [ ]:
results = {s: json.loads((RUN_DIR / f"seed{s}" / "metrics.json").read_text())
           for s in SEEDS if (RUN_DIR / f"seed{s}" / "metrics.json").exists()}
assert results, "Nessun seed completato"
print("Seed completati:", list(results))

rows = []
for s, r in results.items():
    for split in ("dev", "eval"):
        m = r[split]
        rows.append({"seed": s, "split": split, "epoca": r["epoca_migliore"],
                     "EER %": 100 * m["eer"], "IC95 EER %": f"{100 * m['eer_ci95'][0]:.2f}â€“{100 * m['eer_ci95'][1]:.2f}",
                     "minDCF": m["min_dcf"], "actDCF": m["act_dcf"], "Cllr": m["cllr"]})
per_seed = pd.DataFrame(rows)
display(per_seed.round(4))

summary = per_seed.groupby("split")[["EER %", "minDCF", "actDCF", "Cllr"]].agg(["mean", "std"])
display(summary.round(4))

### EER per attacco e per codec (eval)

In [ ]:
def by_group(key):
    df = pd.DataFrame({s: r["eval"][key] for s, r in results.items()})
    return (100 * df).assign(media=lambda d: d.mean(axis=1)).round(2)

attacks, codecs = by_group("eer_per_attacco"), by_group("eer_per_codec")
display(attacks.T)
display(codecs.T)

fig, axes = plt.subplots(1, 2, figsize=(14, 3.5), gridspec_kw={"width_ratios": [2, 1]})
attacks["media"].plot.bar(ax=axes[0], title="EER per attacco â€” eval (%)")
codecs["media"].plot.bar(ax=axes[1], title="EER per codec â€” eval ('-' = nessuno) (%)")
plt.tight_layout()

### Distribuzione dei punteggi (eval, primo seed)

In [ ]:
s0 = next(iter(results))
scores = pd.read_csv(RUN_DIR / f"seed{s0}" / "scores_eval.csv")
fig, ax = plt.subplots(figsize=(7, 3.5))
for label, g in scores.groupby("label"):
    ax.hist(g["score"], bins=80, alpha=0.6, density=True, label=label)
ax.axvline(-np.log(0.95 / 0.5), color="k", ls="--", lw=1, label="soglia actDCF")
ax.set(title=f"Punteggi eval â€” seed {s0}", xlabel="logit bonafide âˆ’ logit spoof"); ax.legend()